# experiments.ipynb 
### Multi-scenario simulator results and Demo for urgent request 

QAOA on five 8-task disaster scenarios. Includes exact/greedy/random baselines,
5-seed full-metrics runs.
Saved report: `multi_scenario_report.json

In [1]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import json
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms import QAOA, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler

In [2]:
def build_scenario_8():
    """
    8 candidate observation opportunities over 3 satellites.
    Story: two active conflicts, a wildfire cluster, and a shipping corridor.
    Each task has a human-readable name and description for the demo UI.
    """
    tasks = [
        {"id": "T0", "sat": "SAT_A", "target": "WarZone_Alpha",
         "name": "War Zone Alpha — infrastructure damage",
         "description": "Conflict-zone imaging for civilian corridor mapping.",
         "t0": 0,  "t1": 3,  "priority": 9, "energy": 3, "gs": "GS_EU"},

        {"id": "T1", "sat": "SAT_A", "target": "ShippingLane_RedSea",
         "name": "Red Sea shipping lane — cargo flow",
         "description": "Maritime traffic imaging to detect blockages.",
         "t0": 2,  "t1": 5,  "priority": 4, "energy": 2, "gs": "GS_EU"},

        {"id": "T2", "sat": "SAT_B", "target": "WarZone_Alpha",
         "name": "War Zone Alpha — second pass",
         "description": "Redundant coverage of same conflict zone.",
         "t0": 4,  "t1": 7,  "priority": 6, "energy": 3, "gs": "GS_US"},

        {"id": "T3", "sat": "SAT_B", "target": "Wildfire_California",
         "name": "California wildfire — fire perimeter",
         "description": "Thermal imaging of active fire front.",
         "t0": 6,  "t1": 9,  "priority": 8, "energy": 2, "gs": "GS_US"},

        {"id": "T4", "sat": "SAT_C", "target": "ShippingLane_RedSea",
         "name": "Red Sea shipping lane — second pass",
         "description": "Redundant maritime coverage.",
         "t0": 1,  "t1": 4,  "priority": 5, "energy": 2, "gs": "GS_ASIA"},

        {"id": "T5", "sat": "SAT_C", "target": "Wildfire_California",
         "name": "California wildfire — burn scar",
         "description": "Post-fire assessment imaging.",
         "t0": 8,  "t1": 11, "priority": 7, "energy": 3, "gs": "GS_ASIA"},

        {"id": "T6", "sat": "SAT_A", "target": "WarZone_Alpha",
         "name": "War Zone Alpha — late pass",
         "description": "Evening coverage of same conflict zone.",
         "t0": 10, "t1": 13, "priority": 6, "energy": 2, "gs": "GS_EU"},

        {"id": "T7", "sat": "SAT_C", "target": "Wildfire_California",
         "name": "California wildfire — final pass",
         "description": "Last scheduled pass over fire zone.",
         "t0": 12, "t1": 15, "priority": 8, "energy": 1, "gs": "GS_ASIA"},
    ]

    conflicts = set()
    for i, a in enumerate(tasks):
        for b in tasks[i+1:]:
            if a["sat"] == b["sat"] and not (a["t1"] <= b["t0"] or b["t1"] <= a["t0"]):
                conflicts.add((a["id"], b["id"]))
            elif a["gs"] == b["gs"] and not (a["t1"] <= b["t0"] or b["t1"] <= a["t0"]):
                conflicts.add((a["id"], b["id"]))
            elif a["target"] == b["target"]:
                conflicts.add((a["id"], b["id"]))
    return tasks, sorted(conflicts)

In [3]:
tasks8, conflicts8 = build_scenario_8()

In [4]:
G = nx.Graph()
G.add_nodes_from([t["id"] for t in tasks8])
G.add_edges_from(conflicts8)

degrees = dict(G.degree())
print("Node degrees:", degrees)
print("Isolated nodes:", [n for n, d in degrees.items() if d == 0])
print("Max degree:", max(degrees.values()))
print("Connected components:", nx.number_connected_components(G))

# Hard instance = no isolated nodes, degrees >= 1, graph is connected
is_hard = all(d >= 1 for d in degrees.values()) and nx.is_connected(G)
print("Non-trivial instance:", is_hard)

Node degrees: {'T0': 3, 'T1': 2, 'T2': 3, 'T3': 3, 'T4': 1, 'T5': 2, 'T6': 2, 'T7': 2}
Isolated nodes: []
Max degree: 3
Connected components: 1
Non-trivial instance: True


In [5]:
def build_qubo_from_scenario(tasks, conflicts, penalty=20.0):
    priorities = {t["id"]: t["priority"] for t in tasks}
    qp = QuadraticProgram(name="DisasterScenario")
    for t in tasks:
        qp.binary_var(name=t["id"])
    qp.minimize(
        linear={k: -v for k, v in priorities.items()},
        quadratic={c: penalty for c in conflicts},
    )
    return qp

qp8 = build_qubo_from_scenario(tasks8, conflicts8)
qubo8 = QuadraticProgramToQubo().convert(qp8)
ising8, _ = qubo8.to_ising()
print("Qubits:", ising8.num_qubits, "Terms:", len(ising8))

Qubits: 8 Terms: 17


In [6]:
def evaluate_schedule(bits_dict, tasks, conflicts):
    """Return (priority, feasible, violations)."""
    prio = sum(t["priority"] for t in tasks if bits_dict.get(t["id"], 0) == 1)
    viol = sum(1 for (a, b) in conflicts if bits_dict.get(a, 0) + bits_dict.get(b, 0) > 1)
    return prio, viol == 0, viol

def exact_solve(qp):
    r = MinimumEigenOptimizer(NumPyMinimumEigensolver()).solve(qp)
    return {v.name: int(round(r.x[i])) for i, v in enumerate(qp.variables)}

def greedy_solve(tasks, conflicts):
    """Sort by priority desc, take if no conflict with already-taken."""
    taken = set()
    for t in sorted(tasks, key=lambda x: -x["priority"]):
        if not any((t["id"], other) in conflicts or (other, t["id"]) in conflicts
                   for other in taken):
            taken.add(t["id"])
    return {t["id"]: int(t["id"] in taken) for t in tasks}

def random_solve(tasks, conflicts, n_samples=1024, seed=0):
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(n_samples):
        bits = {t["id"]: int(rng.integers(0, 2)) for t in tasks}
        p, feas, _ = evaluate_schedule(bits, tasks, conflicts)
        if feas and (best is None or p > best[1]):
            best = (bits, p)
    return best[0] if best else None

exact_bits = exact_solve(qp8)
greedy_bits = greedy_solve(tasks8, conflicts8)
random_bits = random_solve(tasks8, conflicts8, n_samples=1024, seed=0)

for name, bits in [("exact", exact_bits), ("greedy", greedy_bits), ("random", random_bits)]:
    p, feas, v = evaluate_schedule(bits, tasks8, conflicts8)
    print(f"{name:8s} priority={p:3d}  feasible={feas}  violations={v}")

exact    priority= 22  feasible=True  violations=0
greedy   priority= 22  feasible=True  violations=0
random   priority= 22  feasible=True  violations=0


In [7]:
def qaoa_full_metrics(
    qp, tasks, conflicts,
    reps=2, shots=2048, seed=0, maxiter=300
):
    # Reference optimum for THIS scenario.
    reference_bits = exact_solve(qp)
    optimal_priority, reference_feasible, _ = evaluate_schedule(
        reference_bits, tasks, conflicts
    )
    if not reference_feasible:
        raise ValueError("Reference solution is infeasible; check penalties.")

    sampler = StatevectorSampler(
        default_shots=shots,
        seed=seed,
    )
    qaoa = QAOA(
        sampler=sampler,
        optimizer=COBYLA(maxiter=maxiter),
        reps=reps,
        initial_point=np.random.default_rng(seed).uniform(
            0, np.pi, 2 * reps
        ),
    )
    result = MinimumEigenOptimizer(qaoa).solve(qp)

    # Independently sampling the final optimized circuit:::: 
    qc = qaoa.ansatz.assign_parameters(
        result.min_eigen_solver_result.optimal_point
    )
    qc = qc.remove_final_measurements(inplace=False)
    qc.measure_all()

    counts = (
        sampler.run([qc], shots=shots)
        .result()[0]
        .data.meas.get_counts()
    )

    names = [v.name for v in qp.variables]
    feasible_count = 0
    optimal_count = 0
    feasible_priority_sum = 0
    best_feasible_priority = None

    for bitstring, count in counts.items():
        # Qiskit displays qubit 0 at the right.
        values = bitstring.replace(" ", "")[::-1]
        if len(values) != len(names):
            raise ValueError("Unexpected measurement width.")

        decisions = {
            name: int(values[i])
            for i, name in enumerate(names)
        }
        priority, feasible, _ = evaluate_schedule(
            decisions, tasks, conflicts
        )

        if feasible:
            feasible_count += count
            feasible_priority_sum += priority * count

            if priority == optimal_priority:
                optimal_count += count

            if (
                best_feasible_priority is None
                or priority > best_feasible_priority
            ):
                best_feasible_priority = priority

    total = sum(counts.values())

    return {
        "qaoa_best_fval": float(result.fval),
        "qaoa_best_bits": {
            name: int(round(result.x[i]))
            for i, name in enumerate(names)
        },
        "optimal_priority": int(optimal_priority),
        "feasible_prob": feasible_count / total,
        "optimal_prob": optimal_count / total,
        "mean_feasible_priority": (
            feasible_priority_sum / feasible_count
            if feasible_count else None
        ),
        "best_feasible_priority": best_feasible_priority,
        "shots": total,
        "reps": reps,
        "seed": seed,
        "counts": counts,
    }


# Starting with a small smoke test.
metrics = qaoa_full_metrics(
    qp8, tasks8, conflicts8,
    reps=1, shots=512, seed=42, maxiter=30,
)

for key, value in metrics.items():
    if key != "counts":
        print(f"{key}: {value}")

/Users/drsweetyramnani/Documents/Research_Projects/Quantum/qiskit-env/lib/python3.14/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/Users/drsweetyramnani/Documents/Research_Projects/Quantum/qiskit-env/lib/python3.14/site-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/Users/drsweetyramnani/Documents/Research_Projects/Quantum/qiskit-env/lib/python3.14/site-packages/scipy/sparse/_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


qaoa_best_fval: -18.0
qaoa_best_bits: {'T0': 0, 'T1': 0, 'T2': 0, 'T3': 0, 'T4': 1, 'T5': 1, 'T6': 1, 'T7': 0}
optimal_priority: 22
feasible_prob: 0.025390625
optimal_prob: 0.0
mean_feasible_priority: 13.23076923076923
best_feasible_priority: 18
shots: 512
reps: 1
seed: 42


In [8]:
results = []
for seed in range(5):
    m = qaoa_full_metrics(qp8, tasks8, conflicts8, reps=2, shots=2048, seed=seed)
    m["seed"] = seed
    results.append(m)

df = pd.DataFrame(results)
print(df[["seed","feasible_prob","optimal_prob","mean_feasible_priority","best_feasible_priority"]].to_string(index=False))

# Plot
#fig, axes = plt.subplots(1, 3, figsize=(15, 4))
#axes[0].bar(df["seed"], df["feasible_prob"]); axes[0].set_title("Feasible probability")
#axes[1].bar(df["seed"], df["optimal_prob"]);  axes[1].set_title("Optimal probability")
#axes[2].bar(df["seed"], df["mean_feasible_priority"]); axes[2].set_title("Mean priority (feasible)")
#for ax in axes: ax.set_xlabel("seed")
#plt.tight_layout(); plt.savefig("qaoa_metrics.png", dpi=120); plt.show()

 seed  feasible_prob  optimal_prob  mean_feasible_priority  best_feasible_priority
    0       0.612305      0.022461               11.181021                      22
    1       0.351562      0.007324               12.658333                      22
    2       0.434570      0.000000               11.968539                      17
    3       0.254883      0.000977               14.519157                      22
    4       0.270996      0.018066               15.765766                      22


In [9]:
# to display real life scenarios of conflicts 

def generate_conflicts(tasks):
    conflicts = []

    for i, a in enumerate(tasks):
        for b in tasks[i + 1:]:
            overlap = (
                a["t0"] < b["t1"]
                and b["t0"] < a["t1"]
            )
            shared_resource = (
                a["sat"] == b["sat"]
                or a["gs"] == b["gs"]
            )
            same_target = a["target"] == b["target"]

            if (overlap and shared_resource) or same_target:
                conflicts.append((a["id"], b["id"]))

    return sorted(conflicts)


def simulate_disaster_event(
    tasks, previous_schedule, event_time=5
):
    _, feasible, _ = evaluate_schedule(
        previous_schedule, tasks, generate_conflicts(tasks)
    )
    if not feasible:
        raise ValueError("Previous schedule must be feasible.")

    new_task = {
        "id": "T_NEW",
        "sat": "SAT_B",
        "target": "FloodZone_1",
        "t0": event_time,
        "t1": event_time + 3,
        "priority": 10,
        "energy": 3,
        "gs": "GS_US",
    }

    updated_tasks = tasks + [new_task]
    updated_conflicts = generate_conflicts(updated_tasks)

    # Preserving selected AND rejected opportunities whose start
    # time has passed. Selected in-progress tasks cannot be canceled.
    fixed = {
        task["id"]: int(previous_schedule[task["id"]])
        for task in tasks
        if task["t0"] < event_time
    }

    qp_updated = build_qubo_from_scenario(
        updated_tasks, updated_conflicts
    )

    # Substitution preserves interactions with fixed tasks.
    remaining_qp = qp_updated.substitute_variables(
        constants=fixed
    )

    schedule = dict(fixed)
    if remaining_qp.get_num_vars():
        schedule.update(exact_solve(remaining_qp))

    _, feasible, violations = evaluate_schedule(
        schedule, updated_tasks, updated_conflicts
    )
    if not feasible:
        raise ValueError(
            f"Replanned schedule has {violations} violations."
        )

    frozen = [
        task for task in tasks if task["id"] in fixed
    ]
    active = [
        task for task in updated_tasks
        if task["id"] not in fixed
    ]

    return (
        frozen, active, schedule,
        new_task, updated_conflicts,
    )


frozen, active, new_schedule, new_task, updated_conflicts = (
    simulate_disaster_event(tasks8, exact_bits)
)

p_before, _, _ = evaluate_schedule(
    exact_bits, tasks8, conflicts8
)
p_after, feasible_after, violations_after = evaluate_schedule(
    new_schedule,
    tasks8 + [new_task],
    updated_conflicts,
)

print("Replanning solver: exact classical")
print("Fixed decisions:", {
    task["id"]: new_schedule[task["id"]]
    for task in frozen
})
print("Updated schedule:", new_schedule)
print("Priority before:", p_before)
print("Priority after:", p_after)
print("Feasible:", feasible_after)
print("Violations:", violations_after)

assert p_before == 22
assert p_after == 32
assert feasible_after
assert violations_after == 0

Replanning solver: exact classical
Fixed decisions: {'T0': 1, 'T1': 0, 'T2': 0, 'T4': 1}
Updated schedule: {'T0': 1, 'T1': 0, 'T2': 0, 'T4': 1, 'T3': 0, 'T5': 0, 'T6': 0, 'T7': 1, 'T_NEW': 1}
Priority before: 22
Priority after: 32
Feasible: True
Violations: 0


In [10]:
report = {
    "scenario": {"n_tasks": len(tasks8), "n_conflicts": len(conflicts8),
                 "tasks": tasks8, "conflicts": conflicts8},
    "baselines": {
        "exact_priority":  evaluate_schedule(exact_bits,  tasks8, conflicts8)[0],
        "greedy_priority": evaluate_schedule(greedy_bits, tasks8, conflicts8)[0],
        "random_priority": evaluate_schedule(random_bits, tasks8, conflicts8)[0],
    },
    "qaoa_metrics": results,
    "replanning": {"frozen": [t["id"] for t in frozen],
                   "active": [t["id"] for t in active],
                   "before_priority": int(p_before),
                   "after_priority": int(p_after)},
}
with open("scenario_report.json", "w") as f:
    json.dump(report, f, indent=2, default=str)
print("Saved scenario_report.json")

Saved scenario_report.json


In [11]:
from itertools import product

feasible_priorities = []

for values in product([0, 1], repeat=len(tasks8)):
    decisions = dict(zip(
        [task["id"] for task in tasks8],
        values,
    ))
    priority, feasible, _ = evaluate_schedule(
        decisions, tasks8, conflicts8
    )
    if feasible:
        feasible_priorities.append(priority)

optimal_priority = max(feasible_priorities)
number_optimal = feasible_priorities.count(optimal_priority)
total_schedules = 2 ** len(tasks8)

print("Exact optimal priority:", optimal_priority)
print("Number of optimal schedules:", number_optimal)
print("Feasible schedules:", len(feasible_priorities))

print(
    "Uniform feasible probability:",
    len(feasible_priorities) / total_schedules,
)
print(
    "Uniform optimal probability:",
    number_optimal / total_schedules,
)
print(
    "Uniform mean feasible priority:",
    float(np.mean(feasible_priorities)),
)

assert optimal_priority == 22
assert number_optimal == 2
assert len(feasible_priorities) == 41

Exact optimal priority: 22
Number of optimal schedules: 2
Feasible schedules: 41
Uniform feasible probability: 0.16015625
Uniform optimal probability: 0.0078125
Uniform mean feasible priority: 13.317073170731707


Keeping original scenario and generate four synthetic variations with different priorities, times, and shared ground stations. Battery constraints remain outside this experiment.

In [12]:
from copy import deepcopy
from itertools import product
from pathlib import Path
import hashlib
import importlib.metadata
import json
import time

def make_scenario(seed, n_tasks=8):
    rng = np.random.default_rng(seed)
    tasks = []

    for i in range(n_tasks):
        start = int(rng.integers(0, 13))

        tasks.append({
            "id": f"T{i}",
            "sat": str(rng.choice(["SAT_A", "SAT_B", "SAT_C"])),
            "target": str(rng.choice([
                "Flood_North",
                "Flood_South",
                "Wildfire_East",
                "Landslide_West",
            ])),
            "t0": start,
            "t1": start + int(rng.integers(2, 5)),
            "priority": int(rng.integers(1, 13)),
            "gs": str(rng.choice(["GS_1", "GS_2"])),
        })

    return tasks


scenarios = {"original": deepcopy(tasks8)}

for seed in [101, 102, 103, 104]:
    scenarios[f"synthetic_{seed}"] = make_scenario(seed)

for name, tasks in scenarios.items():
    print(
        name,
        "| tasks:", len(tasks),
        "| conflicts:", len(generate_conflicts(tasks)),
    )

original | tasks: 8 | conflicts: 9
synthetic_101 | tasks: 8 | conflicts: 9
synthetic_102 | tasks: 8 | conflicts: 10
synthetic_103 | tasks: 8 | conflicts: 12
synthetic_104 | tasks: 8 | conflicts: 16


## For eight tasks, let me enumerate all 256 schedules.

In [13]:
def scenario_reference(tasks, conflicts):
    names = [task["id"] for task in tasks]
    feasible_scores = []

    for values in product([0, 1], repeat=len(tasks)):
        decisions = dict(zip(names, values))
        score, feasible, _ = evaluate_schedule(
            decisions, tasks, conflicts
        )

        if feasible:
            feasible_scores.append(score)

    optimum = max(feasible_scores)
    total = 2 ** len(tasks)

    greedy_bits = greedy_solve(tasks, conflicts)
    greedy_priority, greedy_feasible, _ = evaluate_schedule(
        greedy_bits, tasks, conflicts
    )
    assert greedy_feasible

    return {
        "optimal_priority": int(optimum),
        "number_optimal": feasible_scores.count(optimum),
        "greedy_priority": int(greedy_priority),
        "uniform_feasible_prob": len(feasible_scores) / total,
        "uniform_optimal_prob": (
            feasible_scores.count(optimum) / total
        ),
        "uniform_mean_feasible_priority": float(
            np.mean(feasible_scores)
        ),
    }


references = {}

for name, tasks in scenarios.items():
    conflicts = generate_conflicts(tasks)
    references[name] = scenario_reference(tasks, conflicts)

display(pd.DataFrame(references).T)

,optimal_priority,number_optimal,greedy_priority,uniform_feasible_prob,uniform_optimal_prob,uniform_mean_feasible_priority
original,22.0,2.0,22.0,0.160156,0.007812,13.317073
synthetic_101,24.0,1.0,24.0,0.175781,0.003906,9.088889
synthetic_102,26.0,1.0,26.0,0.160156,0.003906,13.365854
synthetic_103,25.0,1.0,25.0,0.136719,0.003906,13.542857
synthetic_104,22.0,1.0,22.0,0.089844,0.003906,8.782609


## Running the same experiment on every scenario

### The common penalty of 20 exceeds every task priority in these scenarios. For this conflict-only formulation, removing a selected task from a violated conflict therefore improves the penalized objective, so an exact global minimizer will be feasible.

In [14]:
CONFIG = {
    "reps": 2,
    "shots": 2048,
    "maxiter": 300,
    "penalty": 20.0,
    "seeds": [0, 1, 2, 3, 4],
}

versions = {
    package: importlib.metadata.version(package)
    for package in [
        "qiskit",
        "qiskit-algorithms",
        "qiskit-optimization",
        "numpy",
        "scipy",
    ]
}

records = []

report = {
    "config": CONFIG,
    "versions": versions,
    "model": "Synthetic conflict-only scheduling; no battery constraints",
    "scenarios": {},
    "runs": records,
}

for name, tasks in scenarios.items():
    conflicts = generate_conflicts(tasks)
    reference = references[name]

    instance = {
        "tasks": tasks,
        "conflicts": conflicts,
    }
    fingerprint = hashlib.sha256(
        json.dumps(instance, sort_keys=True).encode()
    ).hexdigest()

    report["scenarios"][name] = {
        **instance,
        "fingerprint": fingerprint,
        "reference": reference,
    }

    qp = build_qubo_from_scenario(
        tasks, conflicts, penalty=CONFIG["penalty"]
    )

    for seed in CONFIG["seeds"]:
        started = time.perf_counter()

        metrics = qaoa_full_metrics(
            qp,
            tasks,
            conflicts,
            reps=CONFIG["reps"],
            shots=CONFIG["shots"],
            maxiter=CONFIG["maxiter"],
            seed=seed,
        )

        assert (
            metrics["optimal_priority"]
            == reference["optimal_priority"]
        )

        # Distinguish the optimizer result from final sampling.
        metrics["optimizer_returned_fval"] = metrics.pop(
            "qaoa_best_fval"
        )
        metrics["optimizer_returned_bits"] = metrics.pop(
            "qaoa_best_bits"
        )

        record = {
            **metrics,
            "scenario": name,
            "elapsed_seconds": time.perf_counter() - started,
            "reference": reference,
        }
        records.append(record)

        Path("multi_scenario_report.json").write_text(
            json.dumps(report, indent=2)
        )

        print(
            f"{name} | seed={seed} | "
            f"feasible={metrics['feasible_prob']:.1%} | "
            f"optimal={metrics['optimal_prob']:.2%} | "
            f"best={metrics['best_feasible_priority']} | "
            f"exact={reference['optimal_priority']}",
            flush=True,
        )

/Users/drsweetyramnani/Documents/Research_Projects/Quantum/qiskit-env/lib/python3.14/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/Users/drsweetyramnani/Documents/Research_Projects/Quantum/qiskit-env/lib/python3.14/site-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/Users/drsweetyramnani/Documents/Research_Projects/Quantum/qiskit-env/lib/python3.14/site-packages/scipy/sparse/_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


original | seed=0 | feasible=61.2% | optimal=2.25% | best=22 | exact=22
original | seed=1 | feasible=35.2% | optimal=0.73% | best=22 | exact=22
original | seed=2 | feasible=43.5% | optimal=0.00% | best=17 | exact=22
original | seed=3 | feasible=25.5% | optimal=0.10% | best=22 | exact=22
original | seed=4 | feasible=27.1% | optimal=1.81% | best=22 | exact=22
synthetic_101 | seed=0 | feasible=22.3% | optimal=0.59% | best=24 | exact=24
synthetic_101 | seed=1 | feasible=55.4% | optimal=0.05% | best=24 | exact=24
synthetic_101 | seed=2 | feasible=40.8% | optimal=1.71% | best=24 | exact=24
synthetic_101 | seed=3 | feasible=22.7% | optimal=0.73% | best=24 | exact=24
synthetic_101 | seed=4 | feasible=34.5% | optimal=0.98% | best=24 | exact=24
synthetic_102 | seed=0 | feasible=19.8% | optimal=0.39% | best=26 | exact=26
synthetic_102 | seed=1 | feasible=16.9% | optimal=2.10% | best=26 | exact=26
synthetic_102 | seed=2 | feasible=32.3% | optimal=0.10% | best=26 | exact=26
synthetic_102 | seed=3 |

### summarizing in this cell 

In [15]:
rows = []

for record in records:
    reference = record["reference"]
    best = record["best_feasible_priority"]

    rows.append({
        "scenario": record["scenario"],
        "seed": record["seed"],
        "feasible_prob": record["feasible_prob"],
        "optimal_prob": record["optimal_prob"],
        "best_priority": best,
        "found_optimum": (
            best == reference["optimal_priority"]
        ),
        "beats_uniform_optimal_prob": (
            record["optimal_prob"]
            > reference["uniform_optimal_prob"]
        ),
    })

runs_df = pd.DataFrame(rows)

summary = runs_df.groupby("scenario").agg(
    runs=("seed", "count"),
    feasible_prob_mean=("feasible_prob", "mean"),
    optimal_prob_mean=("optimal_prob", "mean"),
    optimal_prob_std=("optimal_prob", "std"),
    runs_finding_optimum=("found_optimum", "sum"),
    runs_above_uniform=(
        "beats_uniform_optimal_prob", "sum"
    ),
)

summary = summary.join(
    pd.DataFrame(references).T[
        [
            "optimal_priority",
            "greedy_priority",
            "uniform_optimal_prob",
        ]
    ]
)

display(summary.round(4))
summary.to_csv("multi_scenario_summary.csv")

,runs,feasible_prob_mean,optimal_prob_mean,optimal_prob_std,runs_finding_optimum,runs_above_uniform,optimal_priority,greedy_priority,uniform_optimal_prob
scenario,,,,,,,,,
original,5,0.3849,0.0098,0.0101,4,2,22.0,22.0,0.0078
synthetic_101,5,0.3514,0.0081,0.0061,5,4,24.0,24.0,0.0039
synthetic_102,5,0.2656,0.0056,0.0087,4,1,26.0,26.0,0.0039
synthetic_103,5,0.2549,0.0119,0.0111,4,3,25.0,25.0,0.0039
synthetic_104,5,0.4243,0.0707,0.1529,3,2,22.0,22.0,0.0039


In [17]:
from pathlib import Path
import json
import time

# Separating output so that output stays presevered from previous cells.
output_path = Path("multi_scenario_extended_report.json")

if output_path.exists():
    saved = json.loads(output_path.read_text())
    assert saved["scenarios"] == report["scenarios"]
    assert saved["config"] == CONFIG

    existing = {(r["scenario"], r["seed"]) for r in records}
    for record in saved["runs"]:
        key = (record["scenario"], record["seed"])
        if key not in existing:
            records.append(record)
            existing.add(key)

def save_progress():
    extended_report = {
        **report,
        "runs": records,
        "executed_seeds": sorted({r["seed"] for r in records}),
    }
    temporary_path = output_path.with_suffix(".tmp")
    temporary_path.write_text(
        json.dumps(extended_report, indent=2)
    )
    temporary_path.replace(output_path)

save_progress()
completed = {(r["scenario"], r["seed"]) for r in records}

for name, tasks in scenarios.items():
    conflicts = generate_conflicts(tasks)
    reference = references[name]

    qp = build_qubo_from_scenario(
        tasks, conflicts, penalty=CONFIG["penalty"]
    )

    for seed in range(5, 10):
        if (name, seed) in completed:
            continue

        started = time.perf_counter()
        metrics = qaoa_full_metrics(
            qp, tasks, conflicts,
            reps=CONFIG["reps"],
            shots=CONFIG["shots"],
            maxiter=CONFIG["maxiter"],
            seed=seed,
        )

        assert (
            metrics["optimal_priority"]
            == reference["optimal_priority"]
        )

        metrics["optimizer_returned_fval"] = metrics.pop(
            "qaoa_best_fval"
        )
        metrics["optimizer_returned_bits"] = metrics.pop(
            "qaoa_best_bits"
        )

        records.append({
            **metrics,
            "scenario": name,
            "elapsed_seconds": time.perf_counter() - started,
            "reference": reference,
        })
        completed.add((name, seed))
        save_progress()

        print(
            f"{name} | seed={seed} | "
            f"feasible={metrics['feasible_prob']:.1%} | "
            f"optimal={metrics['optimal_prob']:.2%} | "
            f"best={metrics['best_feasible_priority']}",
            flush=True,
        )

rows = []
for record in records:
    reference = record["reference"]
    rows.append({
        "scenario": record["scenario"],
        "seed": record["seed"],
        "feasible_prob": record["feasible_prob"],
        "optimal_prob": record["optimal_prob"],
        "found_optimum": (
            record["best_feasible_priority"]
            == reference["optimal_priority"]
        ),
        "above_uniform": (
            record["optimal_prob"]
            > reference["uniform_optimal_prob"]
        ),
    })

runs_df = pd.DataFrame(rows)
assert not runs_df.duplicated(["scenario", "seed"]).any()

summary = runs_df.groupby("scenario").agg(
    runs=("seed", "count"),
    feasible_prob_mean=("feasible_prob", "mean"),
    optimal_prob_mean=("optimal_prob", "mean"),
    optimal_prob_median=("optimal_prob", "median"),
    optimal_prob_std=("optimal_prob", "std"),
    runs_finding_optimum=("found_optimum", "sum"),
    runs_above_uniform=("above_uniform", "sum"),
).join(
    pd.DataFrame(references).T[
        ["optimal_priority", "greedy_priority", "uniform_optimal_prob"]
    ]
)

display(summary.round(4))
summary.to_csv("multi_scenario_extended_summary.csv")

/Users/drsweetyramnani/Documents/Research_Projects/Quantum/qiskit-env/lib/python3.14/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/Users/drsweetyramnani/Documents/Research_Projects/Quantum/qiskit-env/lib/python3.14/site-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/Users/drsweetyramnani/Documents/Research_Projects/Quantum/qiskit-env/lib/python3.14/site-packages/scipy/sparse/_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


original | seed=5 | feasible=18.6% | optimal=0.10% | best=22
original | seed=6 | feasible=12.9% | optimal=1.07% | best=22
original | seed=7 | feasible=10.6% | optimal=0.44% | best=22
original | seed=8 | feasible=54.1% | optimal=1.95% | best=22
original | seed=9 | feasible=30.0% | optimal=0.05% | best=22
synthetic_101 | seed=5 | feasible=16.9% | optimal=0.10% | best=24
synthetic_101 | seed=6 | feasible=36.2% | optimal=0.00% | best=22
synthetic_101 | seed=7 | feasible=27.3% | optimal=1.12% | best=24
synthetic_101 | seed=8 | feasible=18.2% | optimal=0.05% | best=24
synthetic_101 | seed=9 | feasible=67.8% | optimal=0.10% | best=24
synthetic_102 | seed=5 | feasible=25.7% | optimal=0.29% | best=26
synthetic_102 | seed=6 | feasible=20.0% | optimal=0.05% | best=26
synthetic_102 | seed=7 | feasible=36.3% | optimal=0.10% | best=26
synthetic_102 | seed=8 | feasible=30.5% | optimal=0.05% | best=26
synthetic_102 | seed=9 | feasible=41.6% | optimal=0.88% | best=26
synthetic_103 | seed=5 | feasible=2

,runs,feasible_prob_mean,optimal_prob_mean,optimal_prob_median,optimal_prob_std,runs_finding_optimum,runs_above_uniform,optimal_priority,greedy_priority,uniform_optimal_prob
scenario,,,,,,,,,,
original,10,0.3186,0.0085,0.0059,0.0087,9,4,22.0,22.0,0.0078
synthetic_101,10,0.3421,0.0054,0.0034,0.0059,9,5,24.0,24.0,0.0039
synthetic_102,10,0.2869,0.0042,0.0015,0.0065,9,2,26.0,26.0,0.0039
synthetic_103,10,0.2588,0.0187,0.0061,0.0349,8,6,25.0,25.0,0.0039
synthetic_104,10,0.3084,0.0401,0.0027,0.1072,8,4,22.0,22.0,0.0039


## Quantum circuit

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

gamma = Parameter("γ")
beta  = Parameter("β")

qc = QuantumCircuit(4, 4)
qc.h(range(4))
qc.barrier(label="Cost")

for p in range(2):  # depth 2
    # cost layer
    for i in range(4):
        qc.rz(gamma, i)
    qc.rzz(gamma, 0, 1)
    qc.rzz(gamma, 2, 3)
    qc.barrier()
    # mixer layer
    for i in range(4):
        qc.rx(2 * beta, i)
    qc.barrier()

qc.measure(range(4), range(4))


qc.draw("mpl", style="iqp", scale=0.8, filename="qaoa_circuit.png")

## Slack free encoding : Qubit reduction 

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

passes   = [4, 6, 8]
standard = [7, 9, 12]
free     = [4, 6, 8]

fig, ax = plt.subplots(figsize=(10, 6), dpi=200)

# the two lines
ax.plot(passes, standard, "o-", color="#94A3B8", linewidth=2.5,
        markersize=10, label="Standard QUBO (Qiskit default)")
ax.plot(passes, free, "o-", color="#14B8A6", linewidth=2.5,
        markersize=10, label="Slack-free (novelty)")

# shade the savings
ax.fill_between(passes, free, standard, color="#14B8A6", alpha=0.15,
                label="Qubits saved")

# annotate each point
for p, s, f in zip(passes, standard, free):
    ax.annotate(f"{s} qubits", (p, s), textcoords="offset points",
                xytext=(0, 12), ha="center", fontsize=11, color="#475569")
    ax.annotate(f"{f} qubits", (p, f), textcoords="offset points",
                xytext=(0, -20), ha="center", fontsize=11,
                fontweight="bold", color="#0F766E")

# annotate the saving percentages in the shaded band
for p, s, f in zip(passes, standard, free):
    pct = round((1 - f/s) * 100)
    ax.annotate(f"−{pct}%", (p, (s + f) / 2), ha="center", va="center",
                fontsize=13, fontweight="bold", color="#0F766E")

ax.set_xlabel("Number of candidate passes", fontsize=12)
ax.set_ylabel("Qubits required", fontsize=12)
ax.set_xticks(passes)
ax.set_ylim(0, 15)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
ax.legend(loc="upper left", fontsize=11, frameon=False)

plt.tight_layout()
plt.savefig("qubit_savings.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(
    "/Users/drsweetyramnani/Documents/Research_Projects/Quantum"
)
OUTPUT = ROOT / "presentation_results"
OUTPUT.mkdir(exist_ok=True)

report = json.loads(
    (ROOT / "multi_scenario_report.json").read_text()
)

labels = {
    "original": "Reference\nscenario",
    "synthetic_101": "Response\nscenario A",
    "synthetic_102": "Response\nscenario B",
    "synthetic_103": "Response\nscenario C",
    "synthetic_104": "Response\nscenario D",
}

rows = []

for name, scenario in report["scenarios"].items():
    runs = [
        run for run in report["runs"]
        if run["scenario"] == name
    ]
    reference = scenario["reference"]
    optimum = reference["optimal_priority"]

    # Do not silently exclude runs with no feasible result.
    best_scores = [
        run["best_feasible_priority"] for run in runs
    ]
    assert all(score is not None for score in best_scores)

    rows.append({
        "label": labels.get(name, name),
        "optimum": optimum,
        "greedy": reference["greedy_priority"],
        "qaoa_mean": np.mean(best_scores),
        "qaoa_std": (
            np.std(best_scores, ddof=1)
            if len(best_scores) > 1 else 0
        ),
        "hits": sum(score == optimum for score in best_scores),
        "runs": len(runs),
    })

df = pd.DataFrame(rows)
x = np.arange(len(df))
width = 0.25

plt.rcParams.update({
    "font.size": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

fig, axes = plt.subplots(
    1, 2, figsize=(15, 6),
    gridspec_kw={"width_ratios": [1.35, 1]},
)

# LEFT: compare delivered schedule scores.
ax = axes[0]

ax.bar(
    x - width, df["optimum"], width,
    label="Exact optimum", color="#94A3B8",
)
ax.bar(
    x, df["greedy"], width,
    label="Classical greedy", color="#14B8A6",
)
ax.bar(
    x + width, df["qaoa_mean"], width,
    yerr=df["qaoa_std"], capsize=4,
    label="QAOA mean best score", color="#2563EB",
)

ax.set_title("How good were the selected schedules?", pad=16)
ax.set_ylabel("Priority points — higher is better")
ax.set_xticks(x, df["label"])
ax.set_ylim(0, max(df["optimum"]) * 1.25)
ax.legend(fontsize=10, loc="upper left")
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

# RIGHT: repeatability across optimizer seeds.
ax = axes[1]
percentages = 100 * df["hits"] / df["runs"]

bars = ax.bar(
    x, percentages, width=0.6, color="#2563EB"
)
ax.bar_label(
    bars,
    labels=[
        f"{hits}/{runs} runs"
        for hits, runs in zip(df["hits"], df["runs"])
    ],
    padding=6,
    fontsize=11,
)

ax.set_title("How often was an optimum found?", pad=16)
ax.set_ylabel("Runs observing an optimal schedule (%)")
ax.set_xticks(x, df["label"])
ax.set_ylim(0, 118)
ax.set_yticks([0, 25, 50, 75, 100])
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

fig.suptitle(
    "Local Qiskit Simulation: Satellite Scheduling Benchmark",
    fontsize=19, fontweight="bold", y=1.02,
)

fig.text(
    0.5, -0.04,
    "Five generated eight-task scenarios; five seeds each; "
    "QAOA depth 2; 2,048 final measurements per run.\n"
    "Error bars: standard deviation across seeds.",
    ha="center", fontsize=10, color="#475569",
)

fig.tight_layout()

fig.savefig(
    OUTPUT / "simulation_benchmark.png",
    dpi=300, bbox_inches="tight",
)
fig.savefig(
    OUTPUT / "simulation_benchmark.svg",
    bbox_inches="tight",
)
plt.show()

print("Saved:", OUTPUT / "simulation_benchmark.png")